In [ ]:
import duckdb
import pandas as pd
# we use np.random for the simulation randomness
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

project_root = Path("..")
cleaned_dir = project_root / "data" / "cleaned"
raw_dir = project_root / "data" / "raw"

# open a mini SQL database session inside Python using duckdb, 
# which allows us to run SQL queries directly on our parquet files 
# without needing to load everything into memory at once
con = duckdb.connect()

fuel_cost_per_mile = 0.20


# ------------------------------------------------------------
# 1. Zone-hour trip statistics
# ------------------------------------------------------------

# part 1 answers: what is a typical trip like if I start in zone i at hour h?

# goal of part 1 is to get number of trips, average net earnings and 
# trip duration by pickup zone and hour, 
# which will be used in the simulation later
# zone_hour_stats will be a dataframe with one row per pickup zone and hour, 
# and columns for the stats we need
zone_hour_stats = con.execute(f"""
SELECT
    t.PULocationID,
    EXTRACT(hour FROM tpep_pickup_datetime) AS pickup_hour,
    z.Borough,
    z.Zone,
    COUNT(*) AS trips,

    AVG(total_amount - {fuel_cost_per_mile} * trip_distance)
        AS avg_net_trip_earnings,

    AVG(
        EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 60.0
    ) AS avg_trip_minutes

FROM read_parquet('{cleaned_dir}/yellow_tripdata_2024-*.parquet') t

LEFT JOIN read_csv_auto('{raw_dir}/taxi_zone_lookup.csv') z
    ON t.PULocationID = z.LocationID

-- keep only trips that start between 8 AM and 8 PM
-- because I want to simulate 8-hour shifts starting at 8 AM or 12 PM
WHERE
    total_amount > 0
    AND z.Borough != 'EWR'
    AND EXTRACT(hour FROM tpep_pickup_datetime) BETWEEN 8 AND 19

GROUP BY
    t.PULocationID,
    pickup_hour,
    z.Borough,
    z.Zone

HAVING COUNT(*) >= 100
""").fetchdf()


# ------------------------------------------------------------
# 2. Zone-hour transitions
# ------------------------------------------------------------

# part 2 answers: where do trips from zone i at hour h typically go?
# zone_hour_transitions is a table with one row per pickup zone, hour, and dropoff zone
# it's not yet a probability table, just counts of trips, but we'll convert it to probabilities later

zone_hour_transitions = con.execute(f"""
SELECT
    PULocationID,
    DOLocationID,
    EXTRACT(hour FROM tpep_pickup_datetime) AS pickup_hour,
    COUNT(*) AS trips

FROM read_parquet('{cleaned_dir}/yellow_tripdata_2024-*.parquet')

WHERE
    total_amount > 0
    AND EXTRACT(hour FROM tpep_pickup_datetime) BETWEEN 8 AND 19

GROUP BY
    PULocationID,
    DOLocationID,
    pickup_hour
""").fetchdf()


# ------------------------------------------------------------
# 3. Restrict to zones with enough data
# ------------------------------------------------------------

# 3 restricts the model to zones with enough data,
# and converts the transition counts to probabilities

# we want to keep zones that have at least 1000 trips in the zone_hour_stats table,
# which should give us enough data to have reliable estimates of the stats and transitions
# this creates a list-like object containiing the zone IDs that meet the threshold
valid_zones = (
    zone_hour_stats.groupby("PULocationID")["trips"]
    # sum all the trips for each pickup zone across all hours to get total trips per zone
    .sum()
    # filter to keep only zones with at least 1000 total trips
    # take the series produced by the previous step
    # call it x, and keep only the values where x >= 1000
    .loc[lambda x: x >= 1000]
    # get the index of the resulting series, which are the valid zone IDs
    .index
)

# filter the zone_hour_stats and zone_hour_transitions tables to keep only rows 
# where the pickup zone is in the valid_zones list
zone_hour_stats = zone_hour_stats[
    zone_hour_stats["PULocationID"].isin(valid_zones)
]

zone_hour_transitions = zone_hour_transitions[
    zone_hour_transitions["PULocationID"].isin(valid_zones)
]

zone_hour_transitions = zone_hour_transitions[
    zone_hour_transitions["DOLocationID"].isin(valid_zones)
]

# this converts the trip counts in zone_hour_transitions to probabilities 
# by dividing each row's trip count by the total trips for that pickup zone and hour
zone_hour_transitions["transition_prob"] = (
    zone_hour_transitions["trips"] /
    zone_hour_transitions.groupby(
        ["PULocationID","pickup_hour"]
    )["trips"].transform("sum")
)


# ------------------------------------------------------------
# 4. Improved wait-time proxy using pickups and dropoffs
# ------------------------------------------------------------

# goal of part 4 is to create a df zone_hour_flows that contains, for each zone and hour, 
# the total pickups, total dropoffs, ratio of dropoffs to pickups,
# and a proxy for expected wait time in minutes based on the ratio of dropoffs to pickups

# first create a new table that aggregates pickups and dropoffs by zone and hour
zone_hour_flows = con.execute(f"""
SELECT
    zone_id,
    hour,
    SUM(pickups) AS pickups,
    SUM(dropoffs) AS dropoffs
FROM (

    SELECT
        PULocationID AS zone_id,
        EXTRACT(hour FROM tpep_pickup_datetime) AS hour,
        COUNT(*) AS pickups,
        0 AS dropoffs
    FROM read_parquet('{cleaned_dir}/yellow_tripdata_2024-*.parquet')
    WHERE
        total_amount > 0
        AND EXTRACT(hour FROM tpep_pickup_datetime) BETWEEN 8 AND 19
    GROUP BY
        PULocationID,
        hour

    UNION ALL

    SELECT
        DOLocationID AS zone_id,
        EXTRACT(hour FROM tpep_dropoff_datetime) AS hour,
        0 AS pickups,
        COUNT(*) AS dropoffs
    FROM read_parquet('{cleaned_dir}/yellow_tripdata_2024-*.parquet')
    WHERE
        total_amount > 0
        AND EXTRACT(hour FROM tpep_dropoff_datetime) BETWEEN 8 AND 19
    GROUP BY
        DOLocationID,
        hour

)
GROUP BY
    zone_id,
    hour
""").fetchdf()

# filter to keep only valid zones that have enough data, which we identified earlier
zone_hour_flows = zone_hour_flows[
    zone_hour_flows["zone_id"].isin(valid_zones)
].copy()

# avoid divide-by-zero
# because we later calculate dropoffs / pickups, we need to make sure pickups is never zero
# this line says if pickups less than 1, set it to 1, otherwise keep it as is
zone_hour_flows["pickups"] = zone_hour_flows["pickups"].clip(lower=1)

# more dropoffs relative to pickups means more taxi competition
zone_hour_flows["dropoff_pickup_ratio"] = (
    zone_hour_flows["dropoffs"] / zone_hour_flows["pickups"]
)

# scaling constant in minutes
wait_scale = 5.0

# the ratio dropoffs / pickups is unitless, so to turn it into minutes 
# we multiply by a scaling constant, which we set to 5 minutes here
zone_hour_flows["expected_wait_minutes"] = (
    wait_scale * zone_hour_flows["dropoff_pickup_ratio"]
)

# keep values in a realistic range of 1 to 30 minutes, to avoid extreme values dominating the simulation
zone_hour_flows["expected_wait_minutes"] = (
    zone_hour_flows["expected_wait_minutes"].clip(lower=1, upper=30)
)

zone_hour_flows.head()


# ------------------------------------------------------------
# 5. Build lookup dictionaries
# ------------------------------------------------------------

# part 5 builds lookup dictionaries from the dataframes we created, 
# which will allow us to quickly access the stats, transitions, and wait times 
# for any given zone and hour during the simulation without needing to do slow dataframe lookups

# given a zone and hour, tell me
# average net earnings for trips starting in that zone and hour
# average trip duration for trips starting in that zone and hour
# zone name and borough for that zone, which we can use for labeling results later
# start with an empty dictionary, and then loop through each row of the zone_hour_stats dataframe
stats_lookup = {}
# _ is the row index, which we don't need, and row is the actual data for that row as a Series
for _, row in zone_hour_stats.iterrows():
    # [(int(row["PULocationID"]), int(row["pickup_hour"]))] is a tuple (zone_id, hour) 
    # that we use as the key in the stats_lookup dictionary
    stats_lookup[(int(row["PULocationID"]), int(row["pickup_hour"]))] = {
        "avg_net_trip_earnings": float(row["avg_net_trip_earnings"]),
        "avg_trip_minutes": float(row["avg_trip_minutes"]),
        "zone_name": row["Zone"],
        "borough": row["Borough"],
    }

# given a zone and hour, tell me the possible destination zones,
# and the probabilities of transitioning to each destination zone based on the historical data
transitions_lookup = {}
# each group is a subset of the zone_hour_transitions dataframe 
# that corresponds to a specific pickup zone and hour
for (pu, hr), group in zone_hour_transitions.groupby(["PULocationID", "pickup_hour"]):
    # extract destination zone IDs as numpy array of integers
    destinations = group["DOLocationID"].astype(int).to_numpy()
    # extract transition probabilities as numpy array of floats
    probabilities = group["transition_prob"].to_numpy(dtype=float)
    # make sure probabilities sum to 1, which is important for the random choice in the simulation
    probabilities = probabilities / probabilities.sum()

    # dictionary uses integer keys for zone and hour
    # but when pandas reads data from duckdb, it might read them as floats, 
    # so we convert to int just to be safe
    transitions_lookup[(int(pu), int(hr))] = {
        "destinations": destinations,
        "probabilities": probabilities,
    }

# given a zone and hour, tell me the expected wait time
wait_lookup = {}
for _, row in zone_hour_flows.iterrows():
    wait_lookup[(int(row["zone_id"]), int(row["hour"]))] = float(row["expected_wait_minutes"])

print("stats keys:", len(stats_lookup))
print("transition keys:", len(transitions_lookup))
print("wait keys:", len(wait_lookup))



# ------------------------------------------------------------
# 6. Simulation
# ------------------------------------------------------------

# this simulates an 8-hour shift starting in a given zone and hour,
# and returns the net earnings per hour and total trip count for that shift
def simulate_shift(start_zone, start_hour=8, shift_minutes=480, rng=None):

    # random number generator for reproducibility, 
    # we can pass in a specific rng with a fixed seed for consistent results across runs
    if rng is None:
        rng = np.random.default_rng()

    current_zone = int(start_zone)

    elapsed_minutes = 0
    total_net_earnings = 0
    trip_count = 0

    # initial waiting time before the first ride
    initial_hour = start_hour
    initial_wait = wait_lookup.get((current_zone, initial_hour), 5)
    elapsed_minutes += initial_wait

    while elapsed_minutes < shift_minutes:

        current_hour = start_hour + int(elapsed_minutes / 60)

        if current_hour > 19:
            break

        stats_key = (current_zone, current_hour)
        trans_key = (current_zone, current_hour)

        if stats_key not in stats_lookup:
            break

        if trans_key not in transitions_lookup:
            break

        trip_stats = stats_lookup[stats_key]

        trip_minutes = trip_stats["avg_trip_minutes"]

        # if zone hour is missing, model assumes a default wait time of 5 minutes before the next trip, 
        # which is a simple way to handle missing data without crashing the simulation
        wait_minutes = wait_lookup.get(
            (current_zone, current_hour),
            5
        )

        total_net_earnings += trip_stats["avg_net_trip_earnings"]

        elapsed_minutes += trip_minutes + wait_minutes

        trip_count += 1

        destinations = transitions_lookup[trans_key]["destinations"]
        probabilities = transitions_lookup[trans_key]["probabilities"]

        # randomly select the next zone
        current_zone = int(
            rng.choice(destinations, p=probabilities)
        )

    hours_worked = elapsed_minutes / 60

    # the function returns a dictionary with the net earnings per hour 
    # and total trip count for the simulated shift
    return {
        "net_earnings_per_hour": total_net_earnings / hours_worked,
        "trip_count": trip_count
    }

# Monte Carlo simulation to evaluate the expected net earnings per hour for starting in each zone at 8 AM,
# by running multiple simulations and averaging the results
def evaluate_start_zone(start_zone, start_hour=8, n_simulations=200):

    # create a list to store the net earnings per hour results from each simulation run
    results = []

    # use a seed because it makes the simulation reproducible
    rng = np.random.default_rng(123)

    # _ means we don't care about the loop index, we just want to run the loop n_simulations times
    for _ in range(n_simulations):

        sim = simulate_shift(
            start_zone=start_zone,
            start_hour=start_hour,
            rng=rng
        )

        results.append(sim["net_earnings_per_hour"])

    # compute the expected earnings, as well as how much the simulated earnings vary across runs, 
    # which gives us a sense of the riskiness of starting in that zone
    return np.nanmean(results), np.nanstd(results)



# ------------------------------------------------------------
# 7. Evaluate zones
# ------------------------------------------------------------

# this loops through each valid zone, runs the simulation to get the expected 
# net earnings per hour and standard deviation for that zone,
# and then looks up the zone name and borough for labeling the results later

# start with an empty list which will store one dictionary per zone
zone_results = []

for zone_id in sorted(valid_zones):

    mean_eph, std_eph = evaluate_start_zone(zone_id)

    zone_name = zone_hour_stats.loc[
        zone_hour_stats.PULocationID == zone_id,
        "Zone"
    ].iloc[0]

    borough = zone_hour_stats.loc[
        zone_hour_stats.PULocationID == zone_id,
        "Borough"
    ].iloc[0]

    # creates python dictionary with the results for this zone, and appends it to the zone_results list
    zone_results.append({

        "PULocationID": zone_id,
        "Zone": zone_name,
        "Borough": borough,
        "expected_net_earnings_per_hour": mean_eph,
        "std_net_earnings_per_hour": std_eph
    })

# convert the list of dictionaries to a DataFrame for easier analysis and visualization
zone_results = pd.DataFrame(zone_results)

zone_results = zone_results.sort_values(
    "expected_net_earnings_per_hour",
    ascending=False
)

zone_results.head()



# ------------------------------------------------------------
# 8. Bar chart of best zones
# ------------------------------------------------------------

import numpy as np

# these correspond to the top 15 best starting zones
# because step 7 sorted the zone_results dataframe by expected net earnings per hour in descending order
top_zones = zone_results.head(15)

# compute standard error, remember we had n=200 simulations for each zone
top_zones["se"] = (
    top_zones["std_net_earnings_per_hour"] / np.sqrt(200)
)

plt.figure(figsize=(10,6))

plt.barh(
    top_zones.Zone,
    top_zones.expected_net_earnings_per_hour,
    xerr=top_zones.se,
    capsize=4
)

# because we want the best zone at the top, we invert the y-axis so it goes from highest to lowest
plt.gca().invert_yaxis()

plt.xlabel("Expected Net Earnings per Hour")

plt.title("Best Starting Zones (8AM Shift)")

plt.show()



# ------------------------------------------------------------
# 9. Map visualization
# ------------------------------------------------------------

zones_map = gpd.read_file(
    project_root / "data/raw/taxi_zones/taxi_zones.shp"
)

map_data = zones_map.merge(

    zone_results,

    left_on="LocationID",

    right_on="PULocationID",

    how="left"
)


fig, ax = plt.subplots(1,1, figsize=(12,10))

map_data.plot(

    column="expected_net_earnings_per_hour",

    cmap="plasma",

    linewidth=0.5,

    edgecolor="black",

    legend=True,

    ax=ax
)

ax.set_title(
    "Expected Net Earnings per Hour by Starting Zone\n(8-Hour Shift Starting at 8 AM)"
)

ax.axis("off")

plt.show()

# ------------------------------------------------------------
# 10. Dot-and-whisker plot
# ------------------------------------------------------------

import numpy as np

top_zones = zone_results.head(15).copy()

# compute standard error
top_zones["se"] = (
    top_zones["std_net_earnings_per_hour"] / np.sqrt(200)
)

plt.figure(figsize=(10,6))

# fmt='o' means we want circular markers for the points, 
# and capsize=4 adds horizontal lines at the ends of the error bars for better visibility
plt.errorbar(
    top_zones.expected_net_earnings_per_hour,
    top_zones.Zone,
    xerr=top_zones.se,
    fmt='o',
    capsize=4
)

plt.gca().invert_yaxis()

plt.xlabel("Expected Net Earnings per Hour")
plt.title("Best Starting Zones (8AM Shift)")

plt.show()

In [ ]:
# export results to csv, will use this for the regression analysis
zone_results.to_csv("../data/zone_results.csv", index=False)

In [ ]:
# histogram of simulated earnings for top zone, zone=93

zone = 93

sim_results = []

for _ in range(500):
    sim = simulate_shift(start_zone=zone)
    sim_results.append(sim["net_earnings_per_hour"])

plt.hist(sim_results, bins=25)

plt.xlabel("Net Earnings per Hour")
plt.ylabel("Frequency")

plt.title("Distribution of Earnings (Start Zone 93)")

plt.show()


In [ ]:
# scatter plot expected earings vs uncertainty

plt.scatter(
    zone_results["expected_net_earnings_per_hour"],
    zone_results["std_net_earnings_per_hour"],
    alpha=0.7
)

plt.xlabel("Expected Earnings per Hour")
plt.ylabel("Std Dev of Earnings")

plt.title("Risk vs Return Across Starting Zones")

plt.show()

In [ ]:
# map expected wait time

fig, ax = plt.subplots(1, 1, figsize=(12,10))

map_wait.plot(
    column="expected_wait_minutes",
    cmap="plasma",
    linewidth=0.5,
    edgecolor="black",
    legend=True,
    ax=ax
)

# remove axis ticks and labels
ax.axis("off")

# add title
ax.set_title("Expected Taxi Wait Time by Zone (8AM–8PM)", fontsize=14)

plt.show()

In [ ]:
print(zone_hour_flows["expected_wait_minutes"].describe())
print(zone_hour_flows["expected_wait_minutes"].min())
print(zone_hour_flows["expected_wait_minutes"].head())

In [ ]:
zone93_transitions = zone_hour_transitions[
    zone_hour_transitions.PULocationID == 93
].sort_values("transition_prob", ascending=False)

zone93_transitions = zone93_transitions.merge(
    zones,
    left_on="DOLocationID",
    right_on="LocationID"
)

In [ ]:
zone93_transitions[["pickup_hour","Zone","transition_prob"]].head(15)

In [ ]:
zone93_8am = zone_hour_transitions[
    (zone_hour_transitions["PULocationID"] == 93) &
    (zone_hour_transitions["pickup_hour"] == 8)
]

zone93_8am = zone93_8am.merge(
    zones,
    left_on="DOLocationID",
    right_on="LocationID",
    how="left"
)

zone93_8am_summary = (
    zone93_8am
    .groupby("Zone")["transition_prob"]
    .sum()
    .sort_values(ascending=False)
)

top_dest = zone93_8am_summary.head(10)

plt.figure(figsize=(10,6))

plt.barh(
    top_dest.index,
    top_dest.values
)

plt.gca().invert_yaxis()

plt.xlabel("Transition Probability")
plt.title("Where Trips from Zone 93 Go (8 AM)")

plt.show()

In [ ]:
# I used the map to figure out which zone number is the yellow one (zone 93)
# now I want the name of that zone so I can label it on the map
zones = con.execute(f"""
SELECT *
FROM read_csv_auto('{raw_dir.as_posix()}/taxi_zone_lookup.csv')
""").fetchdf()

zones[zones["LocationID"] == 93]